Live Deployment Link: <https://diamond-price-prediction-model-the-muqeet.streamlit.app/>

## Setup: Running This Notebook on Google Colab

No installation is needed. Google Colab already has every library this notebook uses (XGBoost is pre-installed on Colab).

1. Go to **https://colab.research.google.com** and sign in with a Google account.
2. Click **File → Upload notebook** and choose this `.ipynb` file.
3. Click **Runtime → Run all**.
4. When **Step 2.1** runs, a **"Choose Files"** button appears below that cell. Click it and select **`Diamonds Prices2022.csv`** from your computer. The notebook continues automatically after the upload.

> **Note:** Colab deletes uploaded files when the runtime is reset or disconnected. If you get a *"file not found"* error, run Step 2.1 again and re-upload the CSV.

---


## Introduction

##### **Definition** A regression ML task that predicts the price (USD) of a diamond from its physical measurements and quality grades.
##### **Application** Jewellery retail pricing, insurance valuation, auction house estimation, and consumer purchasing guidance.

### Aim of This Notebook

> **Which of 5 regression algorithms predicts diamond prices best, and how sure can we be about that answer?**

To answer it fairly we use **K-Fold Cross Validation** and compare the algorithms using **Root Mean Squared Error (RMSE)**.

This notebook uses the **full dataset (53,940 diamonds)** with **10-fold cross validation**.

### What is K-Fold Cross Validation?

A single train/test split can be lucky or unlucky: the result depends on *which* diamonds happened to land in the test set. K-Fold Cross Validation removes this luck:

1. Shuffle the data and split it into **K equal parts (folds)**.
2. Train the model on **K−1 folds** and test it on the **remaining fold**.
3. Repeat **K times**, so that **every fold is used as the test set exactly once**.
4. Report the **average** score over the K rounds (and the standard deviation, to show how much it varies).

With **K = 10** on **53,940 diamonds**, each test fold has about 5,394 diamonds and each training set has about 48,546 diamonds:

```
Round  1:  [TEST ] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train]
Round  2:  [train] | [TEST ] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train]
Round  3:  [train] | [train] | [TEST ] | [train] | [train] | [train] | [train] | [train] | [train] | [train]
  ...
Round 10:  [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [train] | [TEST ]
```

Every diamond is used for testing **exactly once** and for training **K−1 times**.

### Evaluation Measure: Root Mean Squared Error (RMSE)

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

- $y_i$ = actual price, $\hat{y}_i$ = predicted price, $n$ = number of test diamonds
- RMSE is in the **same unit as the output (US dollars)**, so it reads as *"on average, predictions are off by about $X"*.
- **Lower RMSE = better model.** Because errors are squared, RMSE penalises **large mistakes** heavily.

## Input and Output

### Input (Features / Attributes)

| Feature      | Description                                      | Type        | Possible Values              |
|--------------|--------------------------------------------------|-------------|------------------------------|
| **carat**    | Weight of the diamond                            | Numerical   | 0.20 – 5.01                  |
| **depth**    | Total depth percentage: z / mean(x, y) × 100    | Numerical   | 43 – 79                      |
| **table**    | Width of the diamond's top facet (% of widest)  | Numerical   | 43 – 95                      |
| **x**        | Length (mm)                                      | Numerical   | 0 – 10.74                    |
| **y**        | Width (mm)                                       | Numerical   | 0 – 58.9                     |
| **z**        | Depth (mm)                                       | Numerical   | 0 – 31.8                     |
| **cut**      | Cut quality of the diamond                       | Categorical | Fair, Good, Very Good, Premium, Ideal |
| **color**    | Diamond colour grade                             | Categorical | J, I, H, G, F, E, D          |
| **clarity**  | Measurement of diamond clarity                   | Categorical | I1, SI2, SI1, VS2, VS1, VVS2, VVS1, IF |

### Output (Prediction)

| Output Attribute | Description                    | Data Type |
|------------------|--------------------------------|-----------|
| **price**        | Price of the diamond in USD    | Numerical |

### Example Records

| carat | cut     | color | clarity | depth | table | x    | y    | z    | price |
|-------|---------|-------|---------|-------|-------|------|------|------|-------|
| 0.23  | Ideal   | E     | SI2     | 61.5  | 55.0  | 3.95 | 3.98 | 2.43 | 326   |
| 0.21  | Premium | E     | SI1     | 59.8  | 61.0  | 3.89 | 3.84 | 2.31 | 326   |
| 0.70  | Ideal   | G     | VS2     | 61.6  | 56.0  | 5.70 | 5.67 | 3.50 | 2,757 |
| 1.01  | Premium | D     | IF      | 62.5  | 58.0  | 6.37 | 6.30 | 3.96 | 9,523 |

## Developer & System Information

| **Attribute**            | **Details**                                                                  |
|--------------------------|------------------------------------------------------------------------------|
| **Developer Name**       | Student Name, Dr. Rao Muhammad Adeel Nawab                                   |
| **Program Name**         | `diamond_price_kfold_full_dataset_10fold`                                    |
| **IDE**                  | Google Colab                                                                 |
| **Programming Language** | Python 3                                                                     |
| **Libraries**            | NumPy, Pandas, scikit-learn, XGBoost, Matplotlib, tqdm, Pickle (built-in). Exact versions are printed in Step 1. |
| **Dataset**              | `Diamonds Prices2022.csv` (53,940 diamonds, 9 features + price). **CC BY 4.0 License** — Kaggle: https://www.kaggle.com/datasets/nancyalaswad90/diamonds-prices |

### Table of Contents
- **Step 1:** Import Libraries
- **Step 2:** Load Sample Data
  - **Step 2.1:** Upload the Dataset (Google Colab)
  - **Step 2.2:** Read the Dataset
- **Step 3:** Understand and Pre-process Sample Data
  - **Step 3.1:** Understand Sample Data
  - **Step 3.2:** Pre-process Sample Data
- **Step 4:** Feature Extraction
- **Step 5:** Build the Pre-processing Pipeline
- **Step 6:** Define the 5 Regression Algorithms
- **Step 7:** Set Up 10-Fold Cross Validation
  - **Step 7.1:** Create the K-Fold Splitter
  - **Step 7.2:** Inspect the Folds
- **Step 8:** Run 10-Fold Cross Validation for All 5 Algorithms
- **Step 9:** Compare the Algorithms
  - **Step 9.1:** RMSE of Each Algorithm on Each Fold
  - **Step 9.2:** Summary Comparison Table
  - **Step 9.3:** Visual Comparison
  - **Step 9.4:** Observations
- **Step 10:** Train the Final Model
  - **Step 10.1:** Select the Best Algorithm
  - **Step 10.2:** Train the Best Algorithm on All the Data
  - **Step 10.3:** Save the Trained Model
- **Step 11:** Execute the Application Phase
  - **Step 11.1:** Take Input from User
  - **Step 11.2:** Convert User Input into Feature Vector
  - **Step 11.3:** Load the Saved Model
  - **Step 11.4:** Model Prediction
- **Step 12:** Execute the Feedback Phase
- **Conclusion**


# Code — Diamond Price Prediction System
## Step 1: Import Libraries


In [ ]:
# --- Core Python Libraries ---
import os
import pickle
import time

import numpy as np
import pandas as pd

# --- Machine Learning (scikit-learn) ---
import sklearn
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# --- The 5 Regression Algorithms ---
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import ExtraTreesRegressor, AdaBoostRegressor
from xgboost import XGBRegressor

# --- Visualization ---
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator

# --- Progress Bar ---
from tqdm.auto import tqdm

# Print library versions for reproducibility
import xgboost
print(f"scikit-learn : {sklearn.__version__}")
print(f"xgboost      : {xgboost.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")


# Step 2: Load Sample Data
## Step 2.1: Upload the Dataset (Google Colab)

When this cell runs in Google Colab, a **"Choose Files"** button appears. Select **`Diamonds Prices2022.csv`**.

If the file is already present (for example, you uploaded it earlier, or you are running the notebook on your own computer), the upload is skipped.


In [ ]:
DATA_FILE = "Diamonds Prices2022.csv"

# google.colab is only available inside Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists(DATA_FILE):
    print(f"Please click 'Choose Files' and select '{DATA_FILE}' ...")
    uploaded = files.upload()

    # If the browser renamed the file (e.g. 'Diamonds Prices2022 (1).csv'), rename it back
    uploaded_name = list(uploaded.keys())[0]
    if uploaded_name != DATA_FILE:
        os.rename(uploaded_name, DATA_FILE)

if os.path.exists(DATA_FILE):
    print(f"'{DATA_FILE}' is ready to use.")
else:
    raise FileNotFoundError(f"'{DATA_FILE}' not found. Please run this cell again and upload it.")


## Step 2.2: Read the Dataset

This notebook uses **all** the diamonds in the dataset.


In [ ]:
# Load the full diamonds dataset
diamonds = pd.read_csv(DATA_FILE)

print(f"Loaded dataset: {diamonds.shape[0]:,} diamonds, {diamonds.shape[1]} columns")

# Preview the dataset
diamonds.head()


# Step 3: Understand and Pre-process Sample Data
## Step 3.1: Understand Sample Data


In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code helps us understand the dataset by:
1. Printing the names of all attributes (columns).
2. Showing the total number of instances (rows).
3. Showing how many values are missing in each attribute.
"""

print("\nAttributes in Dataset:")
print("======================\n")
print(list(diamonds.columns))

print("\nNumber of Instances in Dataset:")
print("================================\n")
print(len(diamonds))

print("\nMissing Values in Each Attribute:")
print("==================================\n")
missing = diamonds.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")


### Understanding the Output (price)

Before choosing an evaluation measure, it is important to look at the **range of the output**. Diamond prices in this dataset are **right-skewed**: most diamonds cost $1,000–$5,000, but a few rare stones cost up to nearly $19,000.


In [ ]:
# Summary statistics of the diamond price
price_summary = diamonds["price"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
price_summary.to_frame(name="price ($)")


> **Why this matters for RMSE:** RMSE squares each error, so **one badly predicted high-value diamond can raise the RMSE more than hundreds of well-predicted ordinary stones**. Keep this in mind when reading the results: a large RMSE does not necessarily mean the model is bad for *typical* diamonds.


## Step 3.2: Pre-process Sample Data


In [ ]:
"""
Pre-processing steps:
---------------------
1. Drop 'Unnamed: 0' column (unnecessary row index from raw CSV).
2. Remove duplicate rows (identical diamond records). Duplicates are harmful in
   cross validation because a diamond could appear in both the training and test
   folds, making the RMSE look artificially optimistic.
3. Remove rows where x, y, or z equals 0. A diamond cannot have zero dimension in
   any axis; these are measurement errors.
4. Remaining missing numeric values are filled inside the pipeline in Step 5 using
   only training folds.
"""

# 1. Drop index column if present
if "Unnamed: 0" in diamonds.columns:
    diamonds = diamonds.drop(columns=["Unnamed: 0"])
    print("Dropped 'Unnamed: 0' index column.")

# 2. Drop duplicate rows
rows_before = len(diamonds)
diamonds = diamonds.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {rows_before - len(diamonds):,}")

# 3. Remove zero-dimension measurement errors
rows_before = len(diamonds)
diamonds = diamonds[
    (diamonds["x"] != 0) & (diamonds["y"] != 0) & (diamonds["z"] != 0)
].reset_index(drop=True)
print(f"Zero-dimension rows removed: {rows_before - len(diamonds):,}")

print(f"Clean dataset: {len(diamonds):,} diamonds ready for modelling.")


# Step 4: Feature Extraction

We use all **9 input features** from the dataset, and separate the **input (X)** from the **output (y)**.


In [ ]:
numeric_features = ["carat", "depth", "table", "x", "y", "z"]
categorical_features = ["cut", "color", "clarity"]
target = "price"

# Input features (X) and output (y)
X = diamonds[numeric_features + categorical_features]
y = diamonds[target]

print(f"Input features (X): {X.shape[1]} columns -> {list(X.columns)}")
print(f"Output (y)        : {target}")
print(f"Number of diamonds: {len(X):,}")
print(f"Average price     : ${y.mean():,.2f}")

X.head()


# Step 5: Build the Pre-processing Pipeline

Machine learning algorithms need numbers, and linear algorithms (Ridge, Lasso) are sensitive to the scale of each feature. The pipeline follows the exact instructor architecture:

| Feature type | Step | Why |
|---|---|---|
| Numeric (carat, depth, table, x, y, z) | **Fill missing values** with median, then **Standard scaling** (mean 0, std 1) | Algorithms cannot handle empty values; scaling prevents large-range features from dominating |
| Categorical (cut, color, clarity) | **Fill missing values** with most frequent, then **One-hot encoding** | Turns text categories into 0/1 indicator columns |

### Why put pre-processing *inside* the pipeline?

In cross validation, fitting encoders or scalers on the **whole** dataset before splitting causes **data leakage**: the preprocessor would "see" the test fold.

By placing them inside the pipeline, the cross validation loop in Step 8 **re-fits them on the training folds only** in every round, and then applies them to the test fold. This is exactly what happens in the real world, where a model never sees future data during training.

`handle_unknown="ignore"` ensures that unseen categories in a test fold do not crash the pipeline.


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

print("Pre-processing pipeline created:")
preprocessor


# Step 6: Define the 5 Regression Algorithms

We compare five algorithms, from regularised linear models to advanced ensemble methods:

| # | Algorithm | How it predicts a price |
|---|---|---|
| 1 | **Ridge Regression** | Fits a linear relationship with L2 regularisation to avoid overfitting |
| 2 | **Lasso Regression** | Fits a linear relationship with L1 regularisation (automatic feature selection) |
| 3 | **Extra Trees Regressor** | Builds 100 extremely randomized trees and averages their predictions |
| 4 | **AdaBoost Regressor** | Sequentially fits weak decision stumps, focusing on harder examples |
| 5 | **XGBoost Regressor** | Scalable gradient-boosted trees with second-order gradients and regularization |

`random_state=42` makes all results reproducible across runs.


In [ ]:
algorithms = {
    "Ridge Regression": Ridge(alpha=1.0, random_state=42),
    "Lasso Regression": Lasso(alpha=1.0, random_state=42, max_iter=2000),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=100, min_samples_leaf=3, n_jobs=-1, random_state=42
    ),
    "AdaBoost Regressor": AdaBoostRegressor(random_state=42),
    "XGBoost Regressor": XGBRegressor(
        n_estimators=350, learning_rate=0.07, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=42, verbosity=0, eval_metric="rmse"
    ),
}

print("Algorithms to compare:")
for number, name in enumerate(algorithms, start=1):
    print(f"  {number}. {name}")


# Step 7: Set Up 10-Fold Cross Validation
## Step 7.1: Create the K-Fold Splitter

- `n_splits=10`: split the data into 10 folds
- `shuffle=True`: shuffle the diamonds first so folds are representative.
- `random_state=42`: the **same folds are used for all 5 algorithms**. Every algorithm is tested on exactly the same diamonds, which makes the comparison fair.


In [ ]:
K = 10
kfold = KFold(n_splits=K, shuffle=True, random_state=42)

print(f"{K}-Fold Cross Validation created (shuffle=True, random_state=42)")


## Step 7.2: Inspect the Folds

Let's check how many diamonds are used for training and testing in each round.


In [ ]:
fold_sizes = []
for fold_number, (train_index, test_index) in enumerate(kfold.split(X), start=1):
    fold_sizes.append({
        "Fold (Round)": fold_number,
        "Training Diamonds": len(train_index),
        "Testing Diamonds": len(test_index),
        "Avg Price of Test Diamonds ($)": y.iloc[test_index].mean(),
        "Most Expensive Test Diamond ($)": y.iloc[test_index].max(),
    })

fold_sizes_table = pd.DataFrame(fold_sizes).set_index("Fold (Round)")
fold_sizes_table


Notice that the **most expensive test diamond** is different in each fold. A fold that happens to contain very high-value stones will usually have a **higher RMSE**, because of the squaring explained in Step 3.1. With about 5,394 test diamonds per fold the effect is well-balanced across folds.


# Step 8: Run 10-Fold Cross Validation for All 5 Algorithms

For each algorithm we build a pipeline (**pre-processing → algorithm**). Then, for each of the 10 folds, we:

1. make a **fresh copy** of the pipeline (`clone`), so nothing learned in one round is carried into the next,
2. **train** it on the training folds,
3. **predict** the prices of the diamonds in the test fold,
4. calculate the **RMSE** for that fold using `root_mean_squared_error`.

### Progress Bar (tqdm)

`tqdm` shows a **progress bar** while the loop runs. There is one step for each algorithm on each fold, so the bar has **5 × 10 = 50 steps**. While the cell runs, the bar looks like this:

```
XGBoost Regressor, fold 3/10:  44%|████████     | 22/50 [02:15<02:51,  1.10fold/s]
```

| Part of the bar | Meaning |
|---|---|
| `XGBoost Regressor, fold 3/10` | The algorithm and fold being trained right now |
| `44%` and `22/50` | How much of the work is done (steps done / total steps) |
| `02:15` (before `<`) | **Elapsed time**: time spent so far (minutes:seconds) |
| `02:51` (after `<`) | **Remaining time**: estimated time left (minutes:seconds) |
| `1.10fold/s` | Speed: folds per second (or `s/fold` when it is slow) |

> ⏱ **This step takes a few minutes in Colab.** Both XGBoost and Extra Trees use multi-threading (`n_jobs=-1`).


In [ ]:
fold_rmse = {}        # RMSE on each fold, for each algorithm
training_time = {}    # total time taken for all folds

folds = list(kfold.split(X))      # the same folds for every algorithm
total_steps = len(algorithms) * K

with tqdm(total=total_steps, unit="fold") as progress_bar:
    for name, algorithm in algorithms.items():
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("regressor", algorithm),
        ])

        scores = []
        start = time.time()

        for fold_number, (train_index, test_index) in enumerate(folds, start=1):
            progress_bar.set_description(f"{name}, fold {fold_number}/{K}")

            fold_model = clone(model)                         # fresh, untrained copy
            fold_model.fit(X.iloc[train_index], y.iloc[train_index])

            preds = fold_model.predict(X.iloc[test_index])
            rmse = root_mean_squared_error(y.iloc[test_index], preds)
            scores.append(rmse)

            progress_bar.update(1)

        fold_rmse[name] = scores
        training_time[name] = time.time() - start


# Step 9: Compare the Algorithms
## Step 9.1: RMSE of Each Algorithm on Each Fold

Each row is one round of cross validation, and each column is one algorithm. The **lowest RMSE in each row** (the best algorithm on that fold) is highlighted in green.

**How to read it:** count the green cells in each column to see how many folds each algorithm **won**. The algorithm with the best *average* does not always win every fold, which is why we compare the mean over all folds (Step 9.2) rather than trusting any single fold.


In [ ]:
per_fold_table = pd.DataFrame(fold_rmse)
per_fold_table.index = [f"Fold {i}" for i in range(1, K + 1)]
per_fold_table.index.name = "Fold"

(per_fold_table.style
    .format("${:,.0f}")
    .highlight_min(axis=1, props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;"))


## Step 9.2: Summary Comparison Table

This is the main result of the notebook. For each algorithm:

| Column | Meaning |
|---|---|
| **Mean RMSE** | Average RMSE over the 10 folds. This is the **main measure for ranking** the algorithms (lower is better) |
| **Std RMSE** | Standard deviation over the folds. A **small value means the algorithm is consistent**; a large value means its performance depends on which diamonds it gets |
| **Best / Worst Fold** | Lowest and highest RMSE across the folds |
| **RMSE as % of Avg Price** | Mean RMSE divided by the average diamond price. This puts the error in context |
| **Time** | Total time to train and test across all 10 folds |
| **Rank** | 1 = lowest mean RMSE = best algorithm |


In [ ]:
average_price = y.mean()

summary = pd.DataFrame({
    "Mean RMSE ($)": per_fold_table.mean(),
    "Std RMSE ($)": per_fold_table.std(),
    "Best Fold RMSE ($)": per_fold_table.min(),
    "Worst Fold RMSE ($)": per_fold_table.max(),
    "RMSE as % of Avg Price": per_fold_table.mean() / average_price * 100,
    "Time (s)": pd.Series(training_time),
})
summary = summary.sort_values("Mean RMSE ($)")
summary.insert(0, "Rank", range(1, len(summary) + 1))
summary.index.name = "Algorithm"

print(f"Average diamond price in this data: ${average_price:,.2f}\n")

money_columns = ["Mean RMSE ($)", "Std RMSE ($)", "Best Fold RMSE ($)", "Worst Fold RMSE ($)"]
(summary.style
    .format({**{c: "${:,.0f}" for c in money_columns},
             "RMSE as % of Avg Price": "{:.1f}%",
             "Time (s)": "{:.1f}s"})
    .highlight_min(subset=["Mean RMSE ($)"],
                   props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;")
    .highlight_max(subset=["Mean RMSE ($)"],
                   props="background-color: #f9d0d0; color: #0b0b0b;"))


## Step 9.3: Visual Comparison

**Chart 1** shows the mean RMSE of each algorithm (shorter bar = better). The black whisker shows ± 1 standard deviation across the 10 folds.

**Chart 2** shows the RMSE of **every individual fold** as a dot, so you can see how spread out each algorithm's results are.


In [ ]:
# Chart colours — matching the instructor's notebook palette
BAR_COLOR = "#2a78d6"
TEXT_COLOR = "#0b0b0b"
MUTED_TEXT = "#52514e"
GRID_COLOR = "#e4e3df"
dollars = FuncFormatter(lambda value, _: f"${value:,.0f}")

def style_axis(ax):
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.tick_params(colors=MUTED_TEXT, length=0)
    ax.xaxis.set_major_formatter(dollars)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.grid(axis="x", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)

order = summary.index.tolist()            # best algorithm first
positions = np.arange(len(order))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.2))

# ---- Chart 1: mean RMSE with +/- 1 std ----
means = summary["Mean RMSE ($)"].values
stds = summary["Std RMSE ($)"].values

ax1.barh(positions, means, color=BAR_COLOR, height=0.55, zorder=2)
ax1.errorbar(means, positions, xerr=stds, fmt="none",
             color=TEXT_COLOR, capsize=4, linewidth=1.5, zorder=3)

for pos, mean, std in zip(positions, means, stds):
    ax1.text(mean + stds.max() * 0.03, pos, f"${mean:,.0f}",
             va="center", ha="left", color=TEXT_COLOR, fontsize=9)

ax1.set_yticks(positions)
ax1.set_yticklabels(order, color=TEXT_COLOR, fontsize=10)
ax1.invert_yaxis()
ax1.set_xlabel("RMSE (USD)", color=MUTED_TEXT)
ax1.set_title("Mean RMSE (± 1 Std Dev) per Algorithm", color=TEXT_COLOR, fontweight="bold")
style_axis(ax1)

# ---- Chart 2: per-fold RMSE dots ----
for i, name in enumerate(order):
    fold_values = per_fold_table[name].values
    ax2.scatter(fold_values, [i] * K, color=BAR_COLOR, s=40, zorder=2, alpha=0.7)
    ax2.plot([fold_values.min(), fold_values.max()], [i, i],
             color=GRID_COLOR, linewidth=1, zorder=1)

ax2.set_yticks(positions)
ax2.set_yticklabels(order, color=TEXT_COLOR, fontsize=10)
ax2.invert_yaxis()
ax2.set_xlabel("RMSE (USD)", color=MUTED_TEXT)
ax2.set_title("Per-Fold RMSE (each dot = one fold)", color=TEXT_COLOR, fontweight="bold")
style_axis(ax2)

fig.suptitle("Diamond Price Prediction — Algorithm Comparison (10-Fold CV)",
             color=TEXT_COLOR, fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## Step 9.4: Observations

> The numbers below reflect 10-fold cross validation on all 53,940 diamonds.

1. **XGBoost Regressor achieves the lowest mean RMSE (~$520)**, closely followed by Extra Trees (~$524). Both tree ensembles excel at capturing the steep, non-linear jump in diamond prices as carat and quality grades increase.
2. **Ridge and Lasso regression have the highest errors (~$1,128)** — roughly double that of the tree models. Diamond valuation is inherently non-linear; linear models cannot capture the compounding effects of cut, color, and clarity.
3. **AdaBoost performs moderately (~$1,390)**: shallow decision stumps struggle with complex geometric interactions among dimensions (x, y, z) and carat.
4. **Consistency across folds:** With over 5,300 test diamonds per fold, standard deviations are very small (~$15–$25), showing that results are highly reliable and not subject to lucky or unlucky splits.
5. **Practical error rate:** The top algorithms achieve an RMSE of ~13% of average diamond price, providing a solid foundation for automated jewelry valuation.


# Step 10: Train the Final Model
## Step 10.1: Select the Best Algorithm

**Cross validation is used to *choose* an algorithm, not to produce the final model.** The 10 models built during cross validation were only used to measure performance and are now discarded.

We select the algorithm with the **lowest mean RMSE** from the summary table.


In [ ]:
best_algorithm_name = summary.index[0]
best_mean_rmse = summary.loc[best_algorithm_name, "Mean RMSE ($)"]
best_std_rmse = summary.loc[best_algorithm_name, "Std RMSE ($)"]

print(f"Best algorithm : {best_algorithm_name}")
print(f"Expected RMSE  : ${best_mean_rmse:,.2f} (+/- ${best_std_rmse:,.2f})")


## Step 10.2: Train the Best Algorithm on All the Data

Now that we know which algorithm works best, we train it on **all 53,940 diamonds**. There is no separate test set here, because the cross-validation RMSE from Step 9 is already our unbiased estimate of how well this model will perform on new, unseen diamonds.


In [ ]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", algorithms[best_algorithm_name]),
])

final_model.fit(X, y)

print(f"Final {best_algorithm_name} model trained on {len(X):,} diamonds.")


## Step 10.3: Save the Trained Model

The saved file contains the **whole pipeline** (imputer, scaler, one-hot encoder **and** the algorithm), so it can predict directly from raw diamond specifications without any manual preprocessing.

To download `model.pkl` to your computer in Colab, set `DOWNLOAD_MODEL = True` and run the cell again, or use the **Files** panel on the left.

> **Important for deployment:** `model.pkl` must be committed to your GitHub repository so the Streamlit app can load it. See `README.md` for instructions.


In [ ]:
MODEL_FILE = "model.pkl"
DOWNLOAD_MODEL = False

import joblib

# Save with joblib compression (compress=3) to guarantee file size is strictly under 90 MB
joblib.dump(final_model, MODEL_FILE, compress=3)

size_bytes = os.path.getsize(MODEL_FILE)
size_mb = size_bytes / (1024 * 1024)

print(f"Trained model saved as '{MODEL_FILE}'")
print(f"File size: {size_mb:.2f} MB")

if size_mb > 90:
    print("WARNING: model exceeds 90 MB limit.")
else:
    print("✓ File size is within the 90 MB limit.")

if IN_COLAB and DOWNLOAD_MODEL:
    files.download(MODEL_FILE)


# Step 11: Execute the Application Phase
## Step 11.1: Take Input from User

By default the notebook uses the example diamond below, so that **Run all** works without stopping. To type in your own diamond, set `USE_INTERACTIVE_INPUT = True` and run the cell again.


In [ ]:
USE_INTERACTIVE_INPUT = False

# Default example: a 0.9-carat Ideal-cut G-colour VS2-clarity diamond
example_diamond = {
    "carat": 0.90,
    "depth": 61.5,
    "table": 57.0,
    "x": 6.22,
    "y": 6.18,
    "z": 3.81,
    "cut": "Ideal",
    "color": "G",
    "clarity": "VS2",
}

def get_diamond_input():
    print("Enter Diamond Specifications (valid ranges from dataset shown):")
    print(f"  carat   : {X['carat'].min():.2f} - {X['carat'].max():.2f}")
    print(f"  depth   : {X['depth'].min():.1f} - {X['depth'].max():.1f}")
    print(f"  table   : {X['table'].min():.1f} - {X['table'].max():.1f}")
    print(f"  x (mm)  : {X['x'].min():.2f} - {X['x'].max():.2f}")
    print(f"  y (mm)  : {X['y'].min():.2f} - {X['y'].max():.2f}")
    print(f"  z (mm)  : {X['z'].min():.2f} - {X['z'].max():.2f}")
    print("  cut     : Fair, Good, Very Good, Premium, Ideal")
    print("  color   : J, I, H, G, F, E, D")
    print("  clarity : I1, SI2, SI1, VS2, VS1, VVS2, VVS1, IF")
    return {
        "carat": float(input("  Enter carat  : ")),
        "depth": float(input("  Enter depth  : ")),
        "table": float(input("  Enter table  : ")),
        "x": float(input("  Enter x (mm) : ")),
        "y": float(input("  Enter y (mm) : ")),
        "z": float(input("  Enter z (mm) : ")),
        "cut": input("  Enter cut     : ").strip(),
        "color": input("  Enter color   : ").strip(),
        "clarity": input("  Enter clarity : ").strip(),
    }

if USE_INTERACTIVE_INPUT:
    user_input = get_diamond_input()
else:
    user_input = example_diamond
    print("Using example diamond:")
    for k, v in user_input.items():
        print(f"  {k:8s}: {v}")


## Step 11.2: Convert User Input into Feature Vector

The feature vector must have **exactly the same columns, in the same order,** as the training data. There is no need to encode or scale it manually: the saved pipeline handles everything automatically.


In [ ]:
feature_vector = pd.DataFrame([user_input])[numeric_features + categorical_features]
feature_vector


## Step 11.3: Load the Saved Model


In [ ]:
import joblib

try:
    loaded_model = joblib.load(MODEL_FILE)
except Exception:
    with open(MODEL_FILE, "rb") as f:
        loaded_model = pickle.load(f)

print(f"Model loaded from '{MODEL_FILE}': {loaded_model.named_steps['regressor'].__class__.__name__}")


## Step 11.4: Model Prediction

Apply the model to the feature vector of the unseen diamond and return the prediction to the user.


In [ ]:
predicted_price = loaded_model.predict(feature_vector)[0]

print(f"Predicted Price: ${predicted_price:,.2f} USD")
print(f"Typical error (cross-validation RMSE of {best_algorithm_name}): about ${best_mean_rmse:,.0f}")


# Step 12: Execute the Feedback Phase

After deployment, feedback from users and domain experts (jewellers, appraisers) is used to improve the model. Potential improvements for this system:

1. **Feature Engineering**: add volume ($x \times y \times z$) or surface-area features, which directly correlate with raw material size.
2. **Hyperparameter Tuning**: optimize tree depth, learning rate, and regularisation using `RandomizedSearchCV` or Bayesian optimisation across folds.
3. **Target Transformation**: evaluate `log(price)` transformation to better penalize proportional percentage errors across varied diamond weights.
4. **Deployment Monitoring**: log user queries and real-world transactions to detect drift in wholesale vs retail market prices.
5. **App Enhancements**: add batch upload (CSV) so jewelry stores can price complete inventory files in one click.


# Conclusion

In this notebook we compared **5 regression algorithms** on the **full diamonds dataset (53,940 diamonds)** using **10-fold cross validation** and **RMSE**.

- **XGBoost Regressor** achieved the lowest mean RMSE (~$520), followed closely by **Extra Trees** (~$524). XGBoost was selected, retrained on all diamonds, saved as `model.pkl`, and verified inside the application phase.
- **Linear models (Ridge and Lasso) performed worst (~$1,128)**, confirming that diamond valuation has strongly non-linear characteristics that cannot be captured by simple linear equations.
- The 10-fold cross validation setup guarantees that every diamond is used for testing exactly once, yielding statistically sound and dependable evaluation benchmarks.

### Algorithm Ranking Summary

| Rank | Algorithm | Mean RMSE ($) | Std RMSE ($) |
|:---:|---|:---:|:---:|
| 1 | XGBoost Regressor | ~$520 | ~$15 |
| 2 | Extra Trees | ~$524 | ~$16 |
| 3 | Ridge Regression | ~$1,129 | ~$25 |
| 4 | Lasso Regression | ~$1,129 | ~$25 |
| 5 | AdaBoost Regressor | ~$1,390 | ~$35 |
